<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/beginner/03_statistics_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Statistics Fundamentals for Data Scientists

You cannot do DS without statistics. This notebook builds your statistical intuition with code — not just formulas.

**Topics:** Descriptive stats, distributions, CLT simulation, correlation, outlier detection, bootstrapping

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Descriptive Statistics

In [ ]:
# Generate a realistic right-skewed income distribution
n = 1000
income = np.concatenate([
    np.random.normal(45000, 10000, 800),   # most people
    np.random.normal(200000, 50000, 150),  # upper-middle
    np.random.normal(1000000, 200000, 50)  # wealthy
])
income = np.abs(income)  # no negative incomes

# Measures of central tendency
mean = income.mean()
median = np.median(income)
mode_result = stats.mode(income.round(-3))  # round to thousands

print('Central Tendency:')
print(f'  Mean:   ${mean:>12,.0f}  ← pulled up by outliers!')
print(f'  Median: ${median:>12,.0f}  ← better for skewed data')

# Measures of spread
std = income.std()
iqr = np.percentile(income, 75) - np.percentile(income, 25)
cv = std / mean * 100  # coefficient of variation

print('\nSpread:')
print(f'  Std Dev: ${std:>12,.0f}')
print(f'  IQR:     ${iqr:>12,.0f}  ← robust to outliers')
print(f'  CV:       {cv:>11.1f}%   ← normalized spread')

# Shape
skewness = stats.skew(income)
kurtosis = stats.kurtosis(income)
print('\nShape:')
print(f'  Skewness: {skewness:.3f}  (>0 = right-skewed)')
print(f'  Kurtosis: {kurtosis:.3f}  (>0 = heavier tails than Normal)')

# Percentiles
print('\nPercentiles:')
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'  P{p:2d}: ${np.percentile(income, p):>12,.0f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
ax = axes[0]
ax.hist(income / 1000, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(mean/1000, color='red', ls='--', lw=2, label=f'Mean ${mean/1000:.0f}k')
ax.axvline(median/1000, color='green', ls='--', lw=2, label=f'Median ${median/1000:.0f}k')
ax.set_xlabel('Income ($000s)'); ax.set_ylabel('Count')
ax.set_title('Income Distribution (right-skewed)')
ax.legend(); ax.set_xlim(0, 800)

# Box plot
ax2 = axes[1]
ax2.boxplot(income / 1000, vert=True, patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.7))
ax2.set_ylabel('Income ($000s)'); ax2.set_title('Box Plot (shows outliers)')
ax2.set_ylim(0, 800)

# Log-scale histogram (shows structure better for skewed data)
ax3 = axes[2]
ax3.hist(np.log10(income), bins=40, color='darkorange', alpha=0.7, edgecolor='white')
ax3.set_xlabel('log₁₀(Income)'); ax3.set_ylabel('Count')
ax3.set_title('Log-scale: Reveals Normal structure')
ax3.set_xticks([4, 4.5, 5, 5.5, 6])
ax3.set_xticklabels(['$10K', '$32K', '$100K', '$316K', '$1M'])

plt.tight_layout(); plt.show()

## 2. Anscombe's Quartet — Why Visualize?

In [ ]:
# Anscombe's quartet: 4 datasets with identical statistics but completely different patterns
anscombe = sns.load_dataset('anscombe')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, dataset in enumerate(['I', 'II', 'III', 'IV']):
    subset = anscombe[anscombe['dataset'] == dataset]
    ax = axes[idx]
    
    # Same stats for all four
    x, y = subset['x'], subset['y']
    corr = np.corrcoef(x, y)[0, 1]
    slope, intercept, _, _, _ = stats.linregress(x, y)
    
    ax.scatter(x, y, color='steelblue', s=60)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, slope * x_line + intercept, 'r-', lw=2)
    ax.set_title(f'Dataset {dataset}\nμₓ={x.mean():.2f}, μᵧ={y.mean():.2f}\nr={corr:.3f}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_xlim(2, 20); ax.set_ylim(0, 14)

plt.suptitle("Anscombe's Quartet: Same Statistics, Completely Different Patterns!", 
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('LESSON: Always visualize your data before relying on statistics alone!')

## 3. Correlation

In [ ]:
# Generate correlated data
n = 200
x = np.random.randn(n)
y_linear = 2*x + np.random.randn(n) * 0.5        # strong linear
y_noisy  = 0.3*x + np.random.randn(n)             # weak linear
y_quad   = x**2 + np.random.randn(n) * 0.5        # nonlinear
y_indep  = np.random.randn(n)                      # independent

datasets = [
    ('Strong Linear', x, y_linear),
    ('Weak Linear',   x, y_noisy),
    ('Quadratic',     x, y_quad),
    ('Independent',   x, y_indep),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for idx, (name, xi, yi) in enumerate(datasets):
    pearson_r = np.corrcoef(xi, yi)[0, 1]
    spearman_r, _ = stats.spearmanr(xi, yi)
    axes[idx].scatter(xi, yi, alpha=0.4, s=15)
    axes[idx].set_title(f'{name}\nPearson={pearson_r:.3f}\nSpearman={spearman_r:.3f}')
    axes[idx].set_xlabel('x'); axes[idx].set_ylabel('y')

plt.suptitle('Pearson vs Spearman Correlation', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print('Pearson: measures LINEAR correlation → fails for quadratic!')
print('Spearman: measures MONOTONIC correlation → more robust')

## 4. Outlier Detection

In [ ]:
# Three methods for outlier detection
np.random.seed(42)
data = np.concatenate([np.random.normal(50, 10, 97), [120, 150, -20]])  # 3 outliers

# Method 1: Z-score (assumes Normal)
z_scores = np.abs(stats.zscore(data))
outliers_z = data[z_scores > 3]

# Method 2: IQR method (robust)
Q1, Q3 = np.percentile(data, [25, 75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outliers_iqr = data[(data < lower) | (data > upper)]

# Method 3: Modified Z-score (uses median — very robust)
mad = np.median(np.abs(data - np.median(data)))  # Median Absolute Deviation
modified_z = 0.6745 * np.abs(data - np.median(data)) / mad
outliers_mod = data[modified_z > 3.5]

print('Outlier Detection Results:')
print(f'  Z-score (>3σ):   {sorted(outliers_z.round(1))}')
print(f'  IQR (1.5×IQR):   {sorted(outliers_iqr.round(1))}')
print(f'  Modified Z-score:{sorted(outliers_mod.round(1))}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(range(len(data)), np.sort(data), s=20, alpha=0.6, label='Normal data')
outlier_mask = (data < lower) | (data > upper)
ax.scatter(np.where(np.sort(data) > upper)[0], 
           np.sort(data)[np.sort(data) > upper], 
           color='red', s=100, zorder=5, label='IQR outliers')
ax.axhline(upper, color='orange', ls='--', label=f'Upper fence ({upper:.1f})')
ax.axhline(lower, color='orange', ls=':', label=f'Lower fence ({lower:.1f})')
ax.set_title('Outlier Detection with IQR Method')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Bootstrap Confidence Intervals

In [ ]:
# Bootstrap: resample with replacement to estimate any statistic's uncertainty
# Works without distributional assumptions!

np.random.seed(42)
sample = np.random.lognormal(mean=3, sigma=1, size=50)  # skewed data

def bootstrap_ci(data, statistic_fn, n_bootstrap=5000, ci=95):
    """Compute bootstrap confidence interval for any statistic."""
    n = len(data)
    boot_stats = [statistic_fn(np.random.choice(data, n, replace=True)) 
                  for _ in range(n_bootstrap)]
    alpha = (100 - ci) / 2
    return (
        np.percentile(boot_stats, alpha),
        np.percentile(boot_stats, 100 - alpha),
        np.array(boot_stats)
    )

# Bootstrap CI for the mean
lo_mean, hi_mean, boot_means = bootstrap_ci(sample, np.mean)
lo_med, hi_med, boot_meds    = bootstrap_ci(sample, np.median)

print(f'Sample: n={len(sample)}, mean={sample.mean():.2f}, median={np.median(sample):.2f}')
print(f'95% CI for mean:   [{lo_mean:.2f}, {hi_mean:.2f}]')
print(f'95% CI for median: [{lo_med:.2f}, {hi_med:.2f}]')
print('\nBootstrap is powerful: CI for any statistic, no distributional assumptions!')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(boot_means, bins=50, color='steelblue', alpha=0.7, density=True)
ax1.axvline(sample.mean(), color='red', lw=2, label=f'Sample mean={sample.mean():.2f}')
ax1.axvspan(lo_mean, hi_mean, alpha=0.3, color='orange', label=f'95% CI [{lo_mean:.1f}, {hi_mean:.1f}]')
ax1.set_title('Bootstrap Distribution of the Mean'); ax1.legend(fontsize=8)

ax2.hist(boot_meds, bins=50, color='darkorange', alpha=0.7, density=True)
ax2.axvline(np.median(sample), color='red', lw=2, label=f'Sample median={np.median(sample):.2f}')
ax2.axvspan(lo_med, hi_med, alpha=0.3, color='steelblue', label=f'95% CI [{lo_med:.1f}, {hi_med:.1f}]')
ax2.set_title('Bootstrap Distribution of the Median'); ax2.legend(fontsize=8)

plt.tight_layout(); plt.show()